# Encoder Block


In [1]:
from pathlib import Path
import torch
import torch.nn as nn

LECTURE_NOTE_TITLE = 'Encoder Block'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')

class EncoderBlock(nn.Module):
    def __init__(self, d_model=8, n_heads=2, d_ff=32):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x, key_padding_mask=None):
        attn_out, weights = self.attn(x, x, x, key_padding_mask=key_padding_mask, need_weights=True, average_attn_weights=False)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ffn(x))
        return x, weights


Lecture note: Encoder Block


## Encoder block preserves shape


In [2]:
torch.manual_seed(0)
x = torch.randn(2, 4, 8)
block = EncoderBlock()
y, weights = block(x)
print(x.shape, '->', y.shape)
print('weights shape:', weights.shape)


torch.Size([2, 4, 8]) -> torch.Size([2, 4, 8])
weights shape: torch.Size([2, 2, 4, 4])


## Encoder attention is bidirectional


In [3]:
print(weights[0, 0])


tensor([[0.2284, 0.2652, 0.2444, 0.2620],
        [0.2142, 0.2561, 0.2257, 0.3040],
        [0.2281, 0.2413, 0.2742, 0.2564],
        [0.2263, 0.2822, 0.1903, 0.3012]], grad_fn=<SelectBackward0>)


## Padding masks remove invalid tokens


In [4]:
pad_mask = torch.tensor([[False, False, False, True], [False, False, True, True]])
_, masked = block(x, key_padding_mask=pad_mask)
print(masked[0, 0, :, -1])


tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)


## Post-LN and Pre-LN block layouts


In [5]:
class PreLNEncoderBlock(nn.Module):
    def __init__(self, d_model=8, n_heads=2, d_ff=32):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x
print('post-LN sample:', y[0, 0])
print('pre-LN sample :', PreLNEncoderBlock()(x)[0, 0])


post-LN sample: tensor([-0.5727, -0.4206,  0.3803,  0.1699,  1.5288,  0.9003,  0.0561, -2.0422],
       grad_fn=<SelectBackward0>)
pre-LN sample : tensor([-1.8517, -1.1536, -0.0037, -0.1725,  1.3844,  0.0321,  1.2577, -1.9098],
       grad_fn=<SelectBackward0>)


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb
**NANOCHAT**
- nanochat is decoder-only, so there is no direct encoder-block file to pair one-to-one here.
